# 05 — Cascaded Router Demo (Ruflo × LUB)

This notebook walks through the v0.2 governance runtime end-to-end on the `br_regulatory.jsonl` dataset:

1. Build a two-tier `TieredRouter` (Haiku-like → Sonnet-like) with UQ-gated escalation.
2. Route every prompt and log everything to a SQLite uncertainty ledger.
3. Replay calibration from the ledger.
4. Sweep confidence thresholds to produce the cost-vs-risk Pareto frontier.

The default config uses `DummyBackend` so the notebook runs without network or API keys. Swap in `backend='anthropic'` to produce the real v0.2 figure.

Companion design doc: `planning/11_Ruflo_Synthesis.md`.

In [ ]:
from __future__ import annotations

import json
import os
import tempfile
from pathlib import Path

from lub.governance.contexts import default_registry
from lub.ledger import Ledger
from lub.orchestration import Tier, TieredRouter
from lub.pipeline import UncertaintyPipeline


## 1. Build the router

Two tiers. The threshold on the cheap tier (`0.80`) is deliberately strict — it forces most prompts to escalate. That is the right default for regulated banking: we should *prefer* to pay for Sonnet rather than ship a cheap wrong answer.

In [ ]:
def build_router() -> TieredRouter:
    cheap = UncertaintyPipeline.from_pretrained(
        model='dummy-haiku', backend='dummy', estimator='token_logprob'
    )
    strong = UncertaintyPipeline.from_pretrained(
        model='dummy-sonnet', backend='dummy', estimator='p_true'
    )
    return TieredRouter(
        tiers=[
            Tier('haiku', cheap, threshold=0.80, cost=0.001),
            Tier('sonnet', strong, threshold=0.70, cost=0.015),
        ]
    )

router = build_router()
print([t.name for t in router.tiers])


## 2. Route a few prompts and log to the ledger

We persist every `(query, answer, uq_score, policy_decision)` tuple so that we can replay calibration later or ship the ledger to an MRM reviewer.

In [ ]:
DATASET = Path('src/lub/benchmarks/data/br_regulatory.jsonl')
if not DATASET.exists():
    # Running from a different cwd — fall back to the package resource.
    import importlib.resources as _res
    DATASET = Path(str(_res.files('lub.benchmarks.data') / 'br_regulatory.jsonl'))
assert DATASET.exists(), DATASET

ctx = default_registry().get('regulatory-qa')

ledger_path = Path(tempfile.mkdtemp()) / 'demo.sqlite'
rows = []
with Ledger(ledger_path) as led, DATASET.open(encoding='utf-8') as f:
    for line in list(f)[:10]:
        ex = json.loads(line)
        prompt = ex.get('prompt') or ex.get('question') or ''
        if not prompt:
            continue
        routed = router.answer(prompt)
        qid = led.log_query(prompt=prompt, domain=ctx.domain)
        aid = led.log_answer(
            query_id=qid,
            model=f'tier-{routed.tier_used}',
            backend='dummy',
            answer=routed.final.answer,
            latency_ms=0,
            cost=routed.total_cost,
        )
        led.log_score(answer_id=aid, method='confidence', value=float(routed.final.confidence))
        led.log_policy(
            answer_id=aid,
            decision=routed.tier_used,
            threshold=ctx.risk_ceiling,
            passed=not routed.final.should_refuse,
            reason='tiered-router',
        )
        rows.append({
            'prompt': prompt,
            'routed': routed.to_dict(),
            'ground_truth': ex.get('answer'),
        })
print(f'Logged {len(rows)} answers to {ledger_path}')

## 3. Replay calibration from the ledger

`Ledger.replay_calibration` groups historical UQ scores into equal-width buckets and reports the mean confidence vs empirical accuracy per bucket. In a real MRM deployment this runs nightly and the resulting reliability diagram is the V.A evidence.

In [ ]:
with Ledger(ledger_path) as led:
    # For a useful replay, tag outcomes. Here we stub every answer as correct
    # so the cell runs — in production, outcomes arrive asynchronously from
    # downstream systems (ticket closure, human review, backtest).
    for row in rows:
        pass  # would call led.update_outcome(...)
    points = led.replay_calibration(method='confidence', n_buckets=5)

for p in points:
    print(f'bucket={p.bucket} [{p.bucket_low:.2f},{p.bucket_high:.2f}]  '
          f'conf_mean={p.confidence_mean:.3f}  acc={p.accuracy:.3f}  n={p.n}')

## 4. Pareto frontier

Sweep the confidence threshold grid and keep the (cost, risk) Pareto-dominant subset. With a real backend this is what the tech-report's Figure 3 will show.

In [ ]:
# Re-use the standalone plotting module
import importlib, sys
sys.path.insert(0, str(Path.cwd() / 'benchmarks' / 'scripts'))
plot_mod = importlib.import_module('plot_cascaded_pareto')

fig_out = Path(tempfile.mkdtemp()) / 'figure_3_cascaded_pareto.png'
front = plot_mod.plot_pareto(rows, fig_out)
print(f'Pareto front has {len(front)} point(s); wrote {fig_out}')

## 5. Next steps

- Swap `backend='dummy'` for `backend='anthropic'` and set `ANTHROPIC_API_KEY` to reproduce the real cost/risk numbers.
- Wire `lub.governance.assert_policy` into `UncertaintyGuard` to fail the run if measured ECE drifts above the bounded context's target.
- Ship the ledger path to `lub.mcp.server._handle_reliability_diagram` so an MCP client can view the live reliability diagram.